# Module 2: Structured Generation & Extraction Engines

This notebook implements the **SnapAudit ReceiptExtractor Engine** — a bulletproof pipeline that turns **messy OCR receipt text** into **perfectly structured JSON**, safe for SQL insertion.

**What we build:**
1. **Tame LLM randomness** (`temperature=0`, low `top_p`) for deterministic outputs.
2. **Enforce strict data contracts** with Pydantic models.
3. **Self-healing retry loops** — the model sees its own validation errors and auto-corrects.
4. A **`ReceiptExtractor`** engine (Ollama + OpenAI variants).

> **Prerequisites:** [Ollama](https://ollama.com) running locally (for the Ollama path), and an `OPENAI_API_KEY` in your `.env` file (for the OpenAI path).

## 1. Setup: Imports & Environment

All dependencies in one place. We use `python-dotenv` to load secrets from `.env` so we never hard-code API keys.

In [ ]:
# ── Standard library ────────────────────────────────────────────────
import json
import os
import textwrap
from datetime import date
from decimal import Decimal
from typing import Optional, Tuple

# ── Third-party ─────────────────────────────────────────────────────
import requests
from dotenv import load_dotenv  # cleaner than hand-parsing .env
from openai import OpenAI
from pydantic import BaseModel, ConfigDict, Field, ValidationError

# ── Load .env from current directory ──────────────────────
load_dotenv()
env = os.environ

## 2. Data Contract: The `Receipt` Pydantic Model

This schema mirrors what a SQL database expects — types are strict so that downstream `INSERT` statements never crash.

| Field      | Type             | Why                                           |
|------------|------------------|-----------------------------------------------|
| `merchant` | `str`            | Required vendor name                          |
| `date`     | `date \| None`    | ISO `YYYY-MM-DD`; `None` if missing on receipt |
| `currency` | `str`            | 3-letter code like `USD`                      |
| `total`    | `Decimal`        | Exact decimal — `float` would lose precision  |
| `tax`      | `Decimal \| None` | Optional; `None` if not on receipt             |
| `category` | `str`            | High-level label (`Meal`, `Travel`, etc.)     |

In [2]:
class Receipt(BaseModel):
    """Strict data contract for SnapAudit receipts.

    Mirrors the target SQL schema — any mismatch triggers a ValidationError
    that the self-healing loop feeds back to the LLM.
    """

    # arbitrary_types_allowed: lets Pydantic handle Decimal without complaints
    model_config = ConfigDict(arbitrary_types_allowed=True)

    merchant: str = Field(..., description="Name of the merchant or vendor")
    date: Optional[date] = Field(
        default=None,
        description="Transaction date in ISO format YYYY-MM-DD, if present.",
    )
    currency: str = Field(..., description="Three-letter currency code, e.g. 'USD'.")
    # Decimal, not float — avoids floating-point rounding errors in financial data
    total: Decimal = Field(..., description="Total amount charged, as a decimal.")
    tax: Optional[Decimal] = Field(
        default=None, description="Tax amount as a decimal, if inferrable."
    )
    category: str = Field(
        ..., description="High-level category label, e.g. 'Meal', 'Travel'."
    )

a:\PROJECTS\obsidi_AI_academy\.venv\Lib\site-packages\pydantic\_internal\_generate_schema.py:648: ArbitraryTypeWarning: FieldInfo(annotation=NoneType, required=False, default=None, description='Transaction date in ISO format YYYY-MM-DD, if present.') is not a Python type (it may be an instance of an object), Pydantic will allow any object with no validation since we cannot even enforce that the input is an instance of the given type. To get rid of this error wrap the type with `pydantic.SkipValidation`.
  warnings.warn(


## 3. Prompt Template

A **JSON-only contract** shared by both Ollama and OpenAI extractors. The model is forbidden from adding backticks, explanations, or extra keys.

In [3]:
def build_receipt_prompt(receipt_text: str) -> str:
    """Build a strict JSON-only extraction prompt for the Receipt schema."""
    return textwrap.dedent(
        f"""
        You are a strict data extraction engine for financial receipts.

        INPUT: Raw OCR text from a receipt.
        OUTPUT: A single JSON object matching this exact schema:

        {{
          "merchant": string,                     // required
          "date": "YYYY-MM-DD" or null,        // optional
          "currency": string,                    // required, 3-letter code like "USD"
          "total": string,                       // required, decimal with 2 places (e.g. "10.00")
          "tax": string or null,                 // optional, decimal with 2 places
          "category": string                     // required, short label like "Meal" or "Travel"
        }}

        Rules:
        - DO NOT wrap the JSON in backticks or markdown fences.
        - DO NOT include any explanation, comments, or extra keys.
        - Use "null" for missing optional fields — never invent values.
        - Normalize currency amounts to two decimal places as strings.

        --- RECEIPT TEXT START ---
        {receipt_text}
        --- RECEIPT TEXT END ---
        """
    ).strip()

## 4. Base Extractor: Shared Retry Logic

Both the Ollama and OpenAI extractors share the **same** self-healing loop:

1. Call the LLM → get raw text.
2. Parse as JSON → validate against `Receipt`.
3. On failure → feed the error + bad JSON back to the LLM and retry.

We factor this into a **base class** so each provider only needs to override `_call_llm()`.

In [4]:
class BaseReceiptExtractor:
    """Self-healing extraction engine with a shared retry loop.

    Subclasses override `_call_llm(prompt)` to plug in any LLM backend.
    """

    def __init__(self, max_retries: int = 3) -> None:
        self.max_retries = max_retries

    # ── Subclasses must implement this ──────────────────────────────
    def _call_llm(self, prompt: str) -> str:
        raise NotImplementedError

    # ── Clean JSON output by removing markdown code blocks and extra whitespace ───────────────
    def _clean_json_output(self, raw: str) -> str:
        """Clean JSON output by removing markdown code blocks and extra whitespace."""
        raw = raw.strip()
        
        # Remove markdown code blocks if present
        if raw.startswith('```json'):
            raw = raw[7:]  # Remove ```json
        elif raw.startswith('```'):
            raw = raw[3:]   # Remove ```
            
        if raw.endswith('```'):
            raw = raw[:-3]  # Remove closing ```
            
        return raw.strip()

    # ── Parse + validate against Pydantic schema ───────────────
    def _parse_and_validate(self, raw: str) -> Tuple[Receipt, dict]:
        """JSON-parse raw LLM output and validate against `Receipt`."""
        print(f"DEBUG: Raw LLM output: {repr(raw)}")  # Debug output
        cleaned_raw = self._clean_json_output(raw)
        print(f"DEBUG: Cleaned output: {repr(cleaned_raw)}")  # Debug output
        obj = json.loads(cleaned_raw)
        receipt = Receipt.model_validate(obj)
        return receipt, obj

    # ── Self-healing retry loop ────────────────────────────────────
    def extract(self, receipt_text: str) -> Receipt:
        """Extract a `Receipt` from messy OCR text, retrying on validation errors."""
        prompt = build_receipt_prompt(receipt_text)
        last_error: str | None = None
        raw: str = ""

        for attempt in range(1, self.max_retries + 1):
            if attempt == 1:
                raw = self._call_llm(prompt)
            else:
                # Feed the error + bad output back so the model can self-correct
                retry_prompt = (
                    f"The JSON you produced did not pass validation.\n"
                    f"Validation error:\n\n{last_error}\n\n"
                    f"Invalid JSON:\n\n{raw}\n\n"
                    f"Respond with a corrected JSON object only."
                )
                raw = self._call_llm(retry_prompt)

            try:
                receipt, _ = self._parse_and_validate(raw)
                return receipt
            except (json.JSONDecodeError, ValidationError) as e:
                last_error = str(e)
                if attempt == self.max_retries:
                    raise RuntimeError(
                        f"Failed after {self.max_retries} attempts. Last error: {last_error}"
                    ) from e

        raise RuntimeError("Unexpected failure in extract()")  # unreachable

## 5. Ollama Extractor (Local)

Calls a local Ollama model. We pin `temperature=0` and `top_p=0.1` to minimize randomness — critical for structured output.

In [ ]:
OLLAMA_BASE_URL = env.get("OLLAMA_BASE_URL")
OLLAMA_MODEL = env.get("OLLAMA_MODEL")

class ReceiptExtractorOllama(BaseReceiptExtractor):
    """Ollama-backed extractor — runs entirely on your local machine."""

    def __init__(self, model: str = OLLAMA_MODEL, max_retries: int = 3) -> None:
        super().__init__(max_retries=max_retries)
        self.model = model

    def _call_llm(self, prompt: str) -> str:
        # temperature=0 + low top_p → near-deterministic output
        response = requests.post(
            f"{OLLAMA_BASE_URL}/api/chat",
            json={
                "model": self.model,
                "messages": [{"role": "user", "content": prompt}],
                "stream": False,
                "options": {"temperature": 0, "top_p": 0.1},
            },
            timeout=120,
        )
        response.raise_for_status()
        return response.json()["message"]["content"]

### Try it — Ollama

Send a deliberately messy receipt and watch it produce clean, schema-valid JSON.

In [ ]:
if __name__ == "__main__":
    extractor = ReceiptExtractorOllama(model=OLLAMA_MODEL, max_retries=3)

    messy_ocr = """
    SNAPMART GROCERY
    2026/01/17
    ...
    subtotal   $9.5
    TAX 0.50
    TOTAL approximately $10
    Thank you for shopping!!
    """

    receipt = extractor.extract(messy_ocr)
    print("Validated Receipt:")
    print(receipt)
    print("\nAs dict (SQL-ready):")
    print(receipt.model_dump())

DEBUG: Raw LLM output: '{"merchant": "SNAPMART GROCERY", "date": "2026-01-17", "currency": "USD", "total": "10.00", "tax": "0.50", "category": "Meal"}'
DEBUG: Cleaned output: '{"merchant": "SNAPMART GROCERY", "date": "2026-01-17", "currency": "USD", "total": "10.00", "tax": "0.50", "category": "Meal"}'
Validated Receipt:
merchant='SNAPMART GROCERY' date='2026-01-17' currency='USD' total=Decimal('10.00') tax=Decimal('0.50') category='Meal'

As dict (SQL-ready):
{'merchant': 'SNAPMART GROCERY', 'date': '2026-01-17', 'currency': 'USD', 'total': Decimal('10.00'), 'tax': Decimal('0.50'), 'category': 'Meal'}


## 6. OpenAI Extractor

A drop-in replacement using the OpenAI API. Same prompt, same retry loop — only the LLM call differs.

In [7]:
class ReceiptExtractorOpenAI(BaseReceiptExtractor):
    """OpenAI-backed extractor — uses the chat completions API."""

    def __init__(self, model: str = "gpt-4.1-mini", max_retries: int = 3) -> None:
        super().__init__(max_retries=max_retries)
        self.model = model
        self.client = OpenAI()  # reads OPENAI_API_KEY from env

    def _call_llm(self, prompt: str) -> str:
        resp = self.client.chat.completions.create(
            model=self.model,
            temperature=0,
            top_p=0.1,
            messages=[{"role": "user", "content": prompt}],
        )
        return resp.choices[0].message.content

### Try it — OpenAI

Same messy input, different backend. The output should be identically structured.

In [8]:
if __name__ == "__main__":
    oa_extractor = ReceiptExtractorOpenAI(model="gpt-4.1-mini", max_retries=3)

    messy_ocr = """
    CITY CAFE AND BAKERY
    Jan 03 2026
    --
    Latte 4.5
    Muffin 3.25
    Tax   0.64
    Total ten dollars and some cents
    --
    Thank you!
    """

    receipt = oa_extractor.extract(messy_ocr)
    print("Validated Receipt (OpenAI):")
    print(receipt)
    print("\nAs dict (SQL-ready):")
    print(receipt.model_dump())

DEBUG: Raw LLM output: '{\n  "merchant": "CITY CAFE AND BAKERY",\n  "date": "2026-01-03",\n  "currency": "USD",\n  "total": "null",\n  "tax": "0.64",\n  "category": "Meal"\n}'
DEBUG: Cleaned output: '{\n  "merchant": "CITY CAFE AND BAKERY",\n  "date": "2026-01-03",\n  "currency": "USD",\n  "total": "null",\n  "tax": "0.64",\n  "category": "Meal"\n}'
DEBUG: Raw LLM output: '```json\n{\n  "merchant": "CITY CAFE AND BAKERY",\n  "date": "2026-01-03",\n  "currency": "USD",\n  "total": null,\n  "tax": "0.64",\n  "category": "Meal"\n}\n```'
DEBUG: Cleaned output: '{\n  "merchant": "CITY CAFE AND BAKERY",\n  "date": "2026-01-03",\n  "currency": "USD",\n  "total": null,\n  "tax": "0.64",\n  "category": "Meal"\n}'
DEBUG: Raw LLM output: '```json\n{\n  "merchant": "CITY CAFE AND BAKERY",\n  "date": "2026-01-03",\n  "currency": "USD",\n  "total": "0.64",\n  "tax": "0.64",\n  "category": "Meal"\n}\n```'
DEBUG: Cleaned output: '{\n  "merchant": "CITY CAFE AND BAKERY",\n  "date": "2026-01-03",\n  "cu

## Recap

| Concept | How we used it |
|---------|----------------|
| **Deterministic LLM settings** | `temperature=0`, `top_p=0.1` |
| **Pydantic data contracts** | `Receipt` model with `Decimal`, `date`, strict fields |
| **Self-healing retries** | Feed validation errors back to the LLM for auto-correction |
| **DRY base class** | `BaseReceiptExtractor` — swap providers by overriding `_call_llm()` |

**Next step:** Try the interactive Streamlit UI by running:
```bash
streamlit run streamlit_app.py
```